# Ropedia Academy — A2 · 2D & 3D pose estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A2.ipynb)

> **Shows the 2D→3D arc and plots a real keypoint heatmap with its soft-argmax overlaid, plus the recovered 2D skeleton.**
>
> 展示 2D→3D 全流程，并画出真实的关键点热图与叠加其上的 soft-argmax，以及恢复出的 2D 骨架。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A2

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt

H = W = 32
# 2D pose: a per-keypoint HEATMAP -> sub-pixel coordinate (soft-argmax).
def soft_argmax(hm):                           # hm: (K,H,W) score maps
    K = hm.shape[0]
    p = F.softmax(hm.reshape(K, -1), -1).reshape(K, H, W)
    xs, ys = torch.arange(W).float(), torch.arange(H).float()
    return torch.stack([(p.sum(1)*xs).sum(-1), (p.sum(2)*ys).sum(-1)], -1)

# Synthesize 12 keypoint heatmaps as Gaussian blobs at known locations.
gt = torch.rand(12, 2) * torch.tensor([W-1., H-1.])
yy, xx = torch.meshgrid(torch.arange(H).float(), torch.arange(W).float(), indexing='ij')
heatmaps = torch.stack([-(((xx-gx)**2 + (yy-gy)**2)/8) for gx, gy in gt])
kp2d = soft_argmax(heatmaps)

# 3D is ill-posed from one image -> LIFT 2D to 3D with a learned body prior.
lifter = nn.Sequential(nn.Linear(12*2, 128), nn.ReLU(), nn.Linear(128, 12*3))
kp3d = lifter(kp2d.flatten()).reshape(12, 3)
print("2D keypoints:", tuple(kp2d.shape), "-> lifted 3D:", tuple(kp3d.shape))

# Visualize: one heatmap + its soft-argmax, and all recovered 2D keypoints.
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(F.softmax(heatmaps[0].flatten(),0).reshape(H,W), cmap='hot')
ax[0].scatter(*kp2d[0], c='cyan', marker='x', s=80); ax[0].set_title("heatmap + soft-argmax")
ax[1].scatter(*kp2d.T.detach()); ax[1].invert_yaxis(); ax[1].set_title("recovered 2D keypoints")
plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A2
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks